In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

if HF_TOKEN:
    login(HF_TOKEN)
    print("Logged in to Hugging Face!")
else:
    print("HF_TOKEN not found. Please add it in Colab secrets.")

In [ ]:
# Install dependencies
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes

#imports

In [ ]:
# import os
# import torch
# from datasets import load_dataset
# from transformers import (
#     AutoModelForCausalLM,
#     AutoTokenizer,
#     BitsAndBytesConfig,
#     TrainingArguments,
#     Trainer,
#     DataCollatorForLanguageModeling
# )
# from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
# from trl import SFTTrainer


import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import defaultdict
import os
import torch
import numpy as np
import subprocess
import sys
from datasets import load_dataset, Dataset
from transformers import (
AutoModelForCausalLM,
AutoTokenizer,
BitsAndBytesConfig,
TrainingArguments,
Trainer,
DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import warnings
warnings.filterwarnings('ignore')


packages_to_check = {
    'plotly': 'plotly',
    'transformers': 'transformers',
    'datasets': 'datasets'
}

for pkg, pip_name in packages_to_check.items():
    try:
        __import__(pkg)
        print(f"✅ {pkg} already installed")
    except ImportError:
        print(f"⚠️ Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        print(f"✅ {pkg} installed")

# Now import
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    print("✅ Plotly imported")
except Exception as e:
    print(f"❌ Error importing plotly: {e}")
    raise

try:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    print("✅ Transformers imported")
except Exception as e:
    print(f"❌ Error importing transformers: {e}")
    raise

try:
    from datasets import load_dataset, Dataset
    print("✅ Datasets imported")
except Exception as e:
    print(f"❌ Error importing datasets: {e}")
    raise

print()


#CONFIGURATION

In [ ]:
# Model configuration
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
OUTPUT_DIR = "./llama-3.2-3b-finetuned-openwebtext"

# Dataset configuration - Using OpenWebText with streaming
DATASET_NAME = "wikitext" # Changed from "Skylion007/openwebtext" to "wikitext"
MAX_SEQ_LENGTH = 512  # Reduced for T4 GPU memory constraints
DATASET_SPLIT = "train" # wikitext has 'train', 'validation', 'test' splits

# Training configuration
PER_DEVICE_TRAIN_BATCH_SIZE = 1  # Small batch size for T4
GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch size = 4
LEARNING_RATE = 2e-4
NUM_TRAIN_EPOCHS = 1  # Set to 1 for faster completion; increase if needed
MAX_STEPS = 100  # Limit steps for demo; remove or increase for full training
WARMUP_STEPS = 10
LOGGING_STEPS = 10
SAVE_STEPS = 50

# LoRA configuration
LORA_R = 16  # Rank
LORA_ALPHA = 32  # Alpha parameter
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

#4-BIT QUANTIZATION CONFIGURATION

In [ ]:
print("Setting up 4-bit quantization configuration...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",  # NormalFloat 4-bit quantization
    bnb_4bit_compute_dtype=torch.bfloat16,  # Compute dtype for operations
    bnb_4bit_use_double_quant=True,  # Double quantization for memory efficiency
)

#TOKENIZER

In [ ]:
print(f"Loading model: {MODEL_NAME}")
print("This will download the model if not cached...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=True
)

# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "right"  # Required for training

#Model Loading

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
    attn_implementation="eager"
)

#Prepare model for k-bit training

In [ ]:
model = prepare_model_for_kbit_training(model)

# Disable cache for training
model.config.use_cache = True
model.config.pretraining_tp = 1

print(f"Model loaded successfully!")
print(f"Model memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

#CONFIGURE LORA

In [ ]:
print("Configuring LoRA...")

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM"
)

# Get PEFT model
model = get_peft_model(model, lora_config)

#trainable parameters

In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / all_params:.2f}%)")
print(f"All parameters: {all_params:,}")

# LOAD AND PREPARE DATASET WITH STREAMING

In [ ]:
print(f"\nLoading dataset: {DATASET_NAME}")
print("Using STREAMING mode for wikitext...")

# Load dataset with streaming enabled
# wikitext dataset has different configurations, using 'wikitext-103-v1' for simplicity
dataset = load_dataset(
    DATASET_NAME,
    'wikitext-103-v1', # Specify the configuration for wikitext
    split=DATASET_SPLIT,
    streaming=False  # Enable streaming
)

# Tokenization function

In [ ]:
def tokenize_function(examples):
    """Tokenize text data for causal language modeling."""
    # Tokenize the text
    outputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,  # Don't pad here; data collator will handle it
        return_overflowing_tokens=False,
    )
    return outputs

# Apply tokenization to streaming dataset
print("Tokenizing dataset...")
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print(f"Preparing dataset for {MAX_STEPS} training steps...")

# DATA COLLATOR

In [ ]:
# Data collator for causal language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM, not masked LM
)

# TRAINING ARGUMENTS

In [ ]:
print("\nSetting up training arguments...")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_STEPS,  # Limit for demo
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,  # Keep only 2 checkpoints to save disk space
    fp16=False,  # Don't use fp16 with T4
    bf16=True,  # Use bf16 for better stability
    optim="paged_adamw_8bit",  # Memory-efficient optimizer
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="none",  # Disable W&B/tensorboard; set to "tensorboard" if needed
    gradient_checkpointing=True,  # Enable gradient checkpointing to save memory
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
)

# TRAINER

In [ ]:
print("\nInitializing Trainer...")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# TRAINING

# Fine-tuning Llama-3.2-3B on OpenWebText dataset using QLoRA

In [ ]:
print("\n" + "="*80)
print("STARTING TRAINING")
print("="*80)
print(f"Training on T4 GPU with {PER_DEVICE_TRAIN_BATCH_SIZE} batch size")
print(f"Gradient accumulation steps: {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch size: {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Max training steps: {MAX_STEPS}")
print(f"Sequence length: {MAX_SEQ_LENGTH}")
print("="*80 + "\n")

# Check GPU memory before training
if torch.cuda.is_available():
    print(f"GPU Memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"GPU Memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print()

# Train the model
trainer.train()

# SAVE MODEL

In [ ]:
print("\n" + "="*80)
print("TRAINING COMPLETED - SAVING MODEL")
print("="*80)

# Save the fine-tuned model
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Model saved to: {OUTPUT_DIR}")

# Save merged model (optional - combines base model with LoRA adapters)
print("\nTo use the model, load it with:")
print(f"  from peft import PeftModel")
print(f"  base_model = AutoModelForCausalLM.from_pretrained('{MODEL_NAME}')")
print(f"  model = PeftModel.from_pretrained(base_model, '{OUTPUT_DIR}')")


### Preparing .zip of finetuned model and uploading to Google drive

In [ ]:
from google.colab import files
import os

output_zip_file = "/content/llama-3.2-3b-finetuned.zip"
source_dir = "./llama-3.2-3b-finetuned-openwebtext"

# Zip the entire directory
print(f"Zipping directory: {source_dir} to {output_zip_file}")
!zip -r "$output_zip_file" "$source_dir"

# Check if the zip file was created
if os.path.exists(output_zip_file):
    print(f"Initiating download of {output_zip_file}")
    files.download(output_zip_file)
else:
    print(f"Error: Zip file not created at {output_zip_file}")

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
print("Mounting Google Drive...")
try:
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully.")
except Exception as e:
    print(f"❌ Error mounting Google Drive: {e}")

# Define the source file path and destination directory in Drive
source_file_path = "/content/llama-3.2-3b-finetuned.zip"
destination_dir_in_drive = "/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels" # You can change this path

# Ensure the destination directory exists in Drive
if os.path.exists("/content/drive"):
    destination_path = os.path.join(destination_dir_in_drive, os.path.basename(source_file_path))
    os.makedirs(destination_dir_in_drive, exist_ok=True)

    # Check if the source file exists before attempting to copy
    if os.path.exists(source_file_path):
        print(f"\nCopying '{source_file_path}' to '{destination_path}'...")
        try:
            # Use shell command for copying
            !cp "$source_file_path" "$destination_path"
            print("✅ File copied to Google Drive successfully!")
        except Exception as e:
            print(f"❌ Error copying file to Google Drive: {e}")
    else:
        print(f"❌ Error: Source file '{source_file_path}' not found.")
else:
    print("❌ Google Drive not mounted. Cannot copy file.")

#nDNA ANALYSIS

In [ ]:
print("📥 Loading Model...")

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
FINETUNED_MODEL_PATH = "./llama-3.2-3b-finetuned-openwebtext"

model = None
tokenizer = None
model_used = None

In [ ]:
# Try to load fine-tuned model first

print(f"   Attempting to load fine-tuned model from {FINETUNED_MODEL_PATH}...")
from peft import PeftModel

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

# Load LoRA adapters
model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(FINETUNED_MODEL_PATH)
model_used = "meta-llama/Llama-3.2-3B-Instruct (Fine-tuned)"
print("✅ Fine-tuned model loaded successfully!\n")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
)
model_used = "meta-llama/Llama-3.2-3B-Instruct"
print("✅ Base model loaded successfully!\n")


# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"✅ Model: {MODEL_NAME}")
print(f"✅ Vocab Size: {tokenizer.vocab_size}\n")


### METRIC 1 - SPECTRAL CURVATURE (κ)

In [ ]:
def project_tangent(u, v):
    """
    Project vector v onto the tangent space at point u on the unit sphere.
    Projection formula: v_tangent = v - <u,v> * u

    Args:
        u: Point on sphere (batch_size, seq_len, hidden_dim)
        v: Vector to project (batch_size, seq_len, hidden_dim)

    Returns:
        Projected vector on tangent space
    """
    dot_product = torch.sum(u * v, dim=-1, keepdim=True)  # (B, S, 1)
    return v - dot_product * u


def sqrt_embed(q, eps=1e-12):
    """
    Normalize probability distribution to unit-norm embedding on sphere.

    Args:
        q: Probability distribution (batch_size, seq_len, vocab_size)
        eps: Small constant for numerical stability

    Returns:
        Unit-norm embedding u = sqrt(q)
    """
    q_clamped = torch.clamp(q, min=eps)  # Avoid log(0)
    u = torch.sqrt(q_clamped)  # Element-wise sqrt
    u = u / (torch.norm(u, dim=-1, keepdim=True) + eps)  # Normalize to unit sphere
    return u


def compute_spectral_curvature(ulist, eps_curv=1e-12):
    """
    Compute spectral curvature (κ) for a sequence of hidden representations.

    Curvature measures how much the hidden trajectory deviates from geodesics
    on the probability simplex. High curvature = complex non-linear transformations.

    Formula: κ_ℓ = ||d²u|| / (||du|| * ||du||)

    Args:
        ulist: List of unit-norm embeddings at each layer
               [(batch_size, seq_len, hidden_dim), ...]
        eps_curv: Epsilon for numerical stability in curvature denominator

    Returns:
        klist: List of curvature values for interior points
        speeds: List of speed (first derivative norm) at each step
    """
    m = len(ulist)
    assert m >= 3, "Need at least 3 points to compute curvature"

    # Compute first differences (speeds)
    delta_u = []
    speeds = []
    for ell in range(m - 1):
        u_curr = ulist[ell]
        u_next = ulist[ell + 1]
        du = project_tangent(u_curr, u_next - u_curr)  # Project onto tangent space
        delta_u.append(du)

        # Compute speed (norm of velocity vector)
        speed = torch.norm(du, p=2, dim=-1)  # (batch_size, seq_len)
        speeds.append(speed.detach().cpu().numpy())

    # Compute second differences and curvature at interior points
    klist = []
    for ell in range(1, m - 1):
        u_curr = ulist[ell]
        u_next = ulist[ell + 1]
        u_prev = ulist[ell - 1]

        # Second difference: d²u = u_{ℓ+1} - 2*u_ℓ + u_{ℓ-1}
        d2u_raw = u_next - 2 * u_curr + u_prev
        d2u = project_tangent(u_curr, d2u_raw)  # Project to tangent space

        # Numerator: ||d²u||
        num = torch.norm(d2u, p=2, dim=-1)  # (batch_size, seq_len)

        # Denominator: ||du|| * ||du|| (product of adjacent speeds)
        s_prev = torch.norm(delta_u[ell - 1], p=2, dim=-1)  # Speed at ℓ-1 to ℓ
        s_curr = torch.norm(delta_u[ell], p=2, dim=-1)      # Speed at ℓ to ℓ+1

        denom = s_prev * s_curr + eps_curv

        # Curvature
        k_ell = (num / denom).detach().cpu().numpy()  # (batch_size, seq_len)
        klist.append(k_ell)

    return klist, speeds

### PLOT 1 - SPECTRAL CURVATURE

In [ ]:
def plot_spectral_curvature(klist, title="Spectral Curvature (κ) per Layer"):
    """
    Plot spectral curvature across layers.

    Args:
        klist: List of curvature arrays [(B, S), ...]
        title: Plot title
    """
    # Average curvature across batch and sequence
    mean_curvatures = [np.mean(k) for k in klist]

    plt.figure(figsize=(12, 6))
    plt.plot(range(1, len(mean_curvatures) + 1), mean_curvatures, marker='o', linewidth=2)
    plt.title(title, fontsize=14)
    plt.xlabel("Interior Layer Index", fontsize=12)
    plt.ylabel("Mean Spectral Curvature κ", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    return plt.gcf()

### METRIC 2 - THERMODYNAMIC LENGTH (L)

In [ ]:
def compute_fisher_rao_distance(logp_prev, logp_curr, valid_mask=None):
    """
    Compute Fisher-Rao (FR) metric distance between consecutive probability distributions.

    The Fisher-Rao metric is the geodesic distance on the probability simplex.
    Formula: d_FR(p, q) = arccos(Σ√(p_i * q_i))  [Bhattacharyya coefficient]

    Args:
        logp_prev: Log-probabilities at layer ℓ   (batch_size, seq_len, vocab_size)
        logp_curr: Log-probabilities at layer ℓ+1 (batch_size, seq_len, vocab_size)
        valid_mask: Mask for valid positions (batch_size, seq_len)

    Returns:
        fr_steps: Fisher-Rao distance at each position (batch_size, seq_len)
    """
    # Convert log-probs to probabilities
    p_prev = torch.exp(logp_prev)  # (B, S, V)
    p_curr = torch.exp(logp_curr)  # (B, S, V)

    # Bhattacharyya coefficient: BC = Σ sqrt(p_prev * p_curr)
    sqrt_product = torch.sqrt(p_prev * p_curr)  # Element-wise sqrt
    bc = torch.sum(sqrt_product, dim=-1)  # Sum over vocabulary

    # Clamp to [0, 1] for numerical stability
    bc = torch.clamp(bc, 0.0, 1.0)

    # Fisher-Rao distance: 2*arccos(BC)
    fr_steps = 2.0 * torch.acos(bc)  # (B, S)

    # Apply mask if provided
    if valid_mask is not None:
        fr_steps = fr_steps.masked_fill(~valid_mask, 0.0)

    return fr_steps


def compute_thermodynamic_length(model, dataloader, device, max_batches=None):
    """
    Compute thermodynamic length (L) per inter-layer step.

    Thermodynamic length measures the cumulative Fisher-Rao distance across layers.
    Large L indicates complex belief transformations within the model.

    Args:
        model: The transformer model
        dataloader: DataLoader with (input_ids, labels, attention_mask)
        device: torch device (cuda/cpu)
        max_batches: Maximum number of batches to process

    Returns:
        fr_step_means: Mean FR distance for each inter-layer transition
        fr_step_counts: Number of valid tokens per transition
    """
    # Access the transformer layers from the model
    if hasattr(model, 'model') and hasattr(model.model, 'layers'):
        blocks = model.model.layers
    elif hasattr(model, 'transformer') and hasattr(model.transformer, 'h'): # for GPT2-like models
         blocks = model.transformer.h
    else:
        raise AttributeError("Could not find transformer layers in the model.")

    num_steps = len(blocks)
    fr_steps_sums = torch.zeros(num_steps, device=device)
    fr_steps_counts = torch.zeros(num_steps, device=device)

    print(f"Computing thermodynamic length (Fisher-Rao) for {num_steps} inter-layer steps...")

    batch_count = 0
    for batch in tqdm(dataloader, total=max_batches):
        if max_batches and batch_count >= max_batches:
            break
        batch_count += 1

        input_ids = batch['input_ids'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        B, S = input_ids.shape

        # Valid positions: where label != -100
        valid_mask = (labels != -100)  # (B, S)

        # Get initial embeddings (input to the first transformer layer)
        with torch.no_grad():
            # Access embedding and position embedding layers from the main model
            h = model.model.embed_tokens(input_ids) # Llama-specific embedding layer name
            # Check for rotary embeddings (Llama 3 uses these)
            if hasattr(model.model.layers[0], 'self_attn') and hasattr(model.model.layers[0].self_attn, 'rotary_emb'):
                 # Rotary embeddings are applied within the attention mechanism,
                 # so we don't add position embeddings here explicitly like for standard models.
                 pass # No explicit pos embedding to add for Llama 3
            elif hasattr(model.model, 'embed_positions'): # Standard position embeddings
                pos_ids = torch.arange(S, dtype=torch.long, device=device).unsqueeze(0).expand(B, S)
                h = h + model.model.embed_positions(pos_ids)
            else:
                 print("Warning: Could not find standard or rotary position embeddings.")


            # Apply dropout if it exists
            if hasattr(model.model, 'dropout'):
                 h = model.model.dropout(h)
            elif hasattr(model.model, 'embed_dropout'): # Some models might have different names
                 h = model.model.embed_dropout(h)


            hidden_states_list = [h.clone()] # Store initial hidden state

            # Forward pass through blocks and store hidden states after each block
            for ell in range(len(blocks)):
                with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=device.type == 'cuda'):
                    # Pass hidden state through the layer
                    # Need to handle attention mask if it exists in the batch
                    if 'attention_mask' in batch:
                         layer_output = blocks[ell](hidden_states_list[-1], attention_mask=batch['attention_mask'].to(device))[0]
                    else:
                         layer_output = blocks[ell](hidden_states_list[-1])[0]

                    hidden_states_list.append(layer_output.clone())


            # Now compute log probabilities at each step based on hidden states
            # This is an approximation: applying final LN and LM head after each layer's output
            # A more precise method would involve modifying the model forward pass
            logp_list = []
            for hidden_state in hidden_states_list:
                 with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=device.type == 'cuda'):
                    # Apply final LayerNorm and Language Model head
                    # Access these from the main model object
                    logits_ell = model.lm_head(model.model.norm(hidden_state)) # Llama-specific final LN name
                    logp_list.append(F.log_softmax(logits_ell.float(), dim=-1))


            # Compute FR distance between consecutive log probabilities
            for ell in range(len(logp_list) - 1):
                 logp_prev = logp_list[ell]
                 logp_curr = logp_list[ell + 1]
                 fr_steps = compute_fisher_rao_distance(logp_prev, logp_curr, valid_mask)

                 step_idx = ell # FR distance between step ell and ell+1
                 fr_steps_sums[step_idx] += fr_steps.sum()
                 fr_steps_counts[step_idx] += valid_mask.sum().float()


    # Compute mean per step
    fr_step_means = (fr_steps_sums / fr_steps_counts.clamp(min=1)).detach().cpu().numpy()
    fr_step_counts = fr_steps_counts.detach().cpu().numpy()

    return fr_step_means, fr_step_counts

### PLOT- THERMODYNAMIC LENGTH

In [ ]:
def plot_thermodynamic_length(fr_step_means, title="Thermodynamic Length (L) per Inter-Layer Step"):
    """
    Plot thermodynamic length (Fisher-Rao distance) across inter-layer transitions.

    Args:
        fr_step_means: Array of mean FR distances
        title: Plot title
    """
    plt.figure(figsize=(12, 6))
    layer_pairs = range(1, len(fr_step_means) + 1)
    plt.plot(layer_pairs, fr_step_means, marker='s', linewidth=2, color='orange')
    plt.title(title, fontsize=14)
    plt.xlabel("Between Layers (ℓ → ℓ+1)", fontsize=12)
    plt.ylabel("Mean Fisher-Rao Distance (radians)", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    return plt.gcf()

### METRIC 3 - BELIEF VECTOR

In [ ]:
# def compute_belief_vector(logp_prev, logp_curr):
#     """
#     Compute belief vector change (Δθ) between consecutive layers.

#     Belief vector represents the probability distribution in the natural parameter space.
#     Δθ = logp_curr - logp_prev measures the change in beliefs across layers.

#     Args:
#         logp_prev: Log-probabilities at layer ℓ   (batch_size, seq_len, vocab_size)
#         logp_curr: Log-probabilities at layer ℓ+1 (batch_size, seq_len, vocab_size)

#     Returns:
#         delta_theta: Belief vector change (batch_size, seq_len, vocab_size)
#         delta_theta_norm: L2 norm of belief change (batch_size, seq_len)
#     """
#     delta_theta = logp_curr - logp_prev  # Natural parameter space change
#     delta_theta_norm = torch.norm(delta_theta, p=2, dim=-1)  # L2 norm per position

#     return delta_theta, delta_theta_norm



### PLOT - BELIEF VECTOR FIELD

In [ ]:
# def plot_belief_change(delta_norm_means, title="Belief Vector Change (Δθ) per Inter-Layer Step"):
#     """
#     Plot belief vector change norm across inter-layer transitions.

#     Args:
#         delta_norm_means: Array of mean belief change norms
#         title: Plot title
#     """
#     plt.figure(figsize=(12, 6))
#     layer_pairs = range(1, len(delta_norm_means) + 1)
#     plt.plot(layer_pairs, delta_norm_means, marker='^', linewidth=2, color='green')
#     plt.title(title, fontsize=14)
#     plt.xlabel("Between Layers (ℓ → ℓ+1)", fontsize=12)
#     plt.ylabel("Mean ||Δθ|| (L2 norm)", fontsize=12)
#     plt.grid(True, alpha=0.3)
#     plt.tight_layout()
#     return plt.gcf()

# SUMMARY REPORT

In [ ]:
from torch.utils.data import DataLoader

# Create a DataLoader
dataloader = DataLoader(
    tokenized_dataset.with_format("torch"), # Ensure dataset is in PyTorch tensor format
    batch_size=PER_DEVICE_TRAIN_BATCH_SIZE, # Use the same batch size as training
    collate_fn=data_collator # Use the data collator defined earlier
)

print("✅ DataLoader created successfully.")

In [ ]:
#from ndna_metrics import *

# Get your fine-tuned model outputs
# ulist = [sqrt_embed(model.logits[i]) for i in range(num_layers)]

# Calculate all three metrics
# klist, speeds = compute_spectral_curvature(ulist)
# fr_means, fr_counts = compute_thermodynamic_length(blocks, tokenizer, dataloader, device)
# #all_delta_norms = [compute_belief_vector(logp_prev, logp_curr)[1] for ...]

# # Visualize
# plot_spectral_curvature(klist)
# plot_thermodynamic_length(fr_means)
# #plot_belief_change(np.array(all_delta_norms))

#from ndna_metrics import (
#    sqrt_embed,
#    compute_spectral_curvature,
#    compute_thermodynamic_length,
#    plot_spectral_curvature,
#    plot_thermodynamic_length
#)

# Step 1: Get probability distributions from your model
# You need to pass data through the model to get logits
# Assuming you have a dataloader and device defined
# Example:
# dataloader = ...
# device = ...
# model = ...

ulist = []
qlist = []

print("Generating probability distributions from the model...")

# Process a few batches to get representative data
num_batches_to_process = 5 # Adjust as needed

for i, batch in enumerate(dataloader):
    if i >= num_batches_to_process:
        break
    with torch.no_grad():
        input_ids = batch['input_ids'].to(model.device) # Ensure input is on the correct device
        # Get hidden states from each layer
        outputs = model(input_ids, output_hidden_states=True)
        hidden_states = outputs.hidden_states # This is a tuple of hidden states for each layer

        # Get logits from the last layer
        logits = outputs.logits # Logits from the final layer

        # Calculate probability distribution (softmax)
        q = F.softmax(logits, dim=-1) # (B, S, V)
        qlist.append(q.cpu()) # Store on CPU to save GPU memory

        # Calculate unit-norm embeddings for each layer's hidden states
        # We need to compute logits for each layer to get the probability distribution at each layer
        # This requires accessing the model's internal structure, which can vary by model architecture.
        # For Llama models, you can iterate through the layers and apply the final layer norm and linear head.

        layer_ulist = []
        for layer_hidden_state in hidden_states:
            # Apply final layer norm and linear head to get logits for this layer's hidden state
            # NOTE: This is an approximation and might not perfectly reflect the intermediate layer's "output probability"
            # as the final layer norm and lm_head are designed for the final hidden state.
            # A more accurate approach would require modifying the model's forward pass to return intermediate logits.
            intermediate_logits = model.lm_head(model.model.norm(layer_hidden_state))
            intermediate_q = F.softmax(intermediate_logits, dim=-1)
            layer_ulist.append(sqrt_embed(intermediate_q).cpu()) # Store on CPU

        # We'll use the list of ulist per batch, and then concatenate/process later if needed
        # For now, let's just process one batch for demonstration/debugging
        if i == 0:
            ulist = layer_ulist
            break # Process only the first batch for demonstration


# Step 2: Convert to unit-norm embeddings on sphere
# If processing multiple batches, you would need to concatenate qlist and then compute ulist
# For this example, we are using ulist derived from the hidden states of a single batch

# Step 3: Calculate METRIC 1 - Spectral Curvature (κ)
if len(ulist) >= 3:
    print("\nCalculating Spectral Curvature...")
    klist, speeds = compute_spectral_curvature(ulist)
    mean_kappas = [np.mean(k) for k in klist]
    fig1 = plot_spectral_curvature(klist)
    print(f"✓ Spectral Curvature κ computed: {len(mean_kappas)} layers")
else:
    print("\nSkipping Spectral Curvature: Need at least 3 layers/hidden states.")


# Step 4: Calculate METRIC 2 - Thermodynamic Length (L)
# This metric requires iterating through blocks and computing FR distance between layers
# You need to pass the model's blocks to the function
if hasattr(model, 'model') and hasattr(model.model, 'layers'):
    blocks = model.model.layers # Access the list of transformer blocks
    print("\nCalculating Thermodynamic Length...")
    # Need a dataloader that yields batches with 'input_ids', 'labels', etc.
    # Assuming 'dataloader' is already defined and provides the necessary keys.
    # Also need the device the model is on.
    device = model.device # Get the device from the model
    fr_means, fr_counts = compute_thermodynamic_length(
        blocks=blocks,
        tokenizer=tokenizer,
        dataloader=dataloader,
        device=device,
        max_batches=num_batches_to_process # Use the same number of batches
    )
    fig2 = plot_thermodynamic_length(fr_means)
    print(f"✓ Thermodynamic Length L computed: {len(fr_means)} transitions")
else:
    print("\nSkipping Thermodynamic Length: Could not access model layers.")


# Step 5: Calculate METRIC 3 - Belief Vector (Δθ)
# This metric is calculated between consecutive *probability distributions* (logits)
# We have qlist from Step 1
# if len(qlist) >= 2:
#     print("\nCalculating Belief Vector Change...")
#     all_delta_norms = []
#     for i in range(len(qlist) - 1):
#         logp_prev = torch.log(qlist[i] + 1e-12)
#         logp_curr = torch.log(qlist[i+1] + 1e-12)
#         #_, delta_norm = compute_belief_vector(logp_prev, logp_curr)
#         #all_delta_norms.append(np.mean(delta_norm.cpu().numpy()))
#
#     #fig3 = plot_belief_change(np.array(all_delta_norms))
#     #print(f"✓ Belief Vector Δθ computed: {len(all_delta_norms)} transitions")
# else:
#     #print("\nSkipping Belief Vector: Need at least 2 probability distributions.")
#     pass # Belief vector calculation is commented out based on previous cells


print("\nAnalysis Complete.")